In [ ]:
"""
relevance.py

Handles Contextual Relevance Scoring (Semantic Re-ranking) by querying
the local Master Database over specific time constraints.
"""

import os
import json
import sqlite3
from collections import defaultdict

import numpy as np
import pandas as pd
from tqdm import tqdm

# Relative package imports
from .data_ops import parse_date_robust

# <<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>
# HELPER FUNCTIONS
# <<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>

def _extract_base_terms(mesh_str: str) -> set:
    """Safely extracts base MeSH terms by stripping asterisks and subheadings."""
    if not mesh_str or not isinstance(mesh_str, str):
        return set()
    bases = set()
    for t in mesh_str.split(';'):
        base = t.split('/')[0].lstrip('*').strip()
        if base:
            bases.add(base)
    return bases


# <<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>
# CORE PIPELINE OPERATIONS
# <<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>

def run_contextual_relevance_scoring(input_nodes_file: str, output_nodes_file: str,
                                     master_db_path: str, relevance_db_path: str,
                                     id_key: str, weight_key_1: str, final_key_1: str,
                                     weight_key_2: str, final_key_2: str,
                                     start_date_param: str, end_date_param: str,
                                     entrez_email: str = "", entrez_api_key: str = "",
                                     calculate_full_centrality: bool = True):
    """
    Calculates contextual relevance by querying the local Master Database
    over the specific time constraints, completely bypassing API limits.
    """
    if not calculate_full_centrality:
        print("\n[!] NOTICE: User skipped centrality. CRS scores now reflect TOPIC DENSITY (term overlap), not topological relevance.\n")

    print(f"\n<<< Loading Seed Terms from {os.path.basename(input_nodes_file)} >>>")
    seed_weights_1, total_weight_1, seed_terms = {}, 0, set()
    seed_weights_2, total_weight_2 = {}, 0

    try:
        with open(input_nodes_file, 'r') as f:
            nodes = json.load(f).get('elements', {}).get('nodes', [])
        for node in nodes:
            data = node.get('data', {})
            term, weight_1, weight_2 = data.get(id_key), data.get(weight_key_1), data.get(weight_key_2)
            if term:
                seed_terms.add(term)
                if isinstance(weight_1, (int, float)):
                    seed_weights_1[term] = float(weight_1)
                    total_weight_1 += float(weight_1)
                if isinstance(weight_2, (int, float)):
                    seed_weights_2[term] = float(weight_2)
                    total_weight_2 += float(weight_2)
        print(f"Loaded {len(seed_terms)} unique seed terms for relevance scoring.")
    except Exception as e:
        raise RuntimeError(f"ERROR loading seed terms: {e}")

    if not seed_terms:
        return

    # <<< Enforce ISO Dates for High-Speed SQLite Query >>>
    start_iso = parse_date_robust(start_date_param).strftime('%Y-%m-%d')
    end_iso = parse_date_robust(end_date_param).strftime('%Y-%m-%d')

    print("\n" + "<"*30 + ">"*30)
    print("<<< Calculating Contextual Relevance (Local Data Lake) >>>")
    print("<"*30 + ">"*30)
    print(f"  Target Date Range: {start_iso} to {end_iso}")

    try:
        # Connect in Read-Only mode to prevent FUSE lock issues
        master_conn = sqlite3.connect(f'file:{master_db_path}?mode=ro', uri=True)
        m_cursor = master_conn.cursor()

        # Schema compatibility check
        m_cursor.execute("PRAGMA table_info(master_mesh_annotations)")
        columns = [info[1] for info in m_cursor.fetchall()]
        if 'pub_date' not in columns:
            raise RuntimeError("CRITICAL ERROR: 'pub_date' column is missing from the Master Database. You MUST run Step 0 again to rebuild the database with the new 4-column schema before proceeding.")

        # Count total articles in range for progress bar
        print(f"  Scanning 30-million record database for temporal matches...")
        m_cursor.execute("SELECT count(*) FROM master_mesh_annotations WHERE pub_date BETWEEN ? AND ?", (start_iso, end_iso))
        total_articles_in_range = m_cursor.fetchone()[0]

        if total_articles_in_range == 0:
            print("  [!] Warning: Zero articles found in Master Database for this date range.")
            master_conn.close()
            return

        print(f"  Found {total_articles_in_range:,} articles matching date constraints. Commencing semantic scoring...")

        m_cursor.execute("SELECT pmid, mesh_terms FROM master_mesh_annotations WHERE pub_date BETWEEN ? AND ?", (start_iso, end_iso))

        aggregator_1 = defaultdict(lambda: {'sum': 0.0, 'count': 0})
        aggregator_2 = defaultdict(lambda: {'sum': 0.0, 'count': 0})
        article_scores_data = []

        # Iterate via fetchmany to control RAM
        chunk_size = 100000
        pbar = tqdm(total=total_articles_in_range, desc="Scoring Articles")

        while True:
            rows = m_cursor.fetchmany(chunk_size)
            if not rows:
                break

            for row in rows:
                pmid, mesh_terms_str = row
                if not mesh_terms_str:
                    continue

                article_terms = _extract_base_terms(mesh_terms_str)
                matching_seeds = article_terms.intersection(seed_terms)

                if not matching_seeds:
                    continue

                score_1, score_2 = 0.0, 0.0
                if total_weight_1 > 0:
                    score_1 = sum(seed_weights_1.get(term, 0) for term in matching_seeds) / total_weight_1
                    for term in matching_seeds:
                        aggregator_1[term]['sum'] += score_1
                        aggregator_1[term]['count'] += 1

                if total_weight_2 > 0:
                    score_2 = sum(seed_weights_2.get(term, 0) for term in matching_seeds) / total_weight_2
                    for term in matching_seeds:
                        aggregator_2[term]['sum'] += score_2
                        aggregator_2[term]['count'] += 1

                article_scores_data.append({
                    'pmid': pmid,
                    f'score_{weight_key_1}': score_1,
                    f'score_{weight_key_2}': score_2,
                    'contributing_seeds': ';'.join(sorted(list(matching_seeds)))
                })

            pbar.update(len(rows))

        pbar.close()
        master_conn.close()

    except Exception as e:
        raise RuntimeError(f"ERROR calculating scores from Master DB: {e}")

    print("\n<<< Saving Individual Article Relevance Scores to Database >>>")
    if article_scores_data:
        scores_df = pd.DataFrame(article_scores_data)
        try:
            os.makedirs(os.path.dirname(relevance_db_path), exist_ok=True)
            conn = sqlite3.connect(relevance_db_path)
            scores_df.to_sql('article_relevance_scores', conn, if_exists='replace', index=False)
            cursor = conn.cursor()
            cursor.execute("CREATE INDEX IF NOT EXISTS idx_pmid ON article_relevance_scores (pmid)")
            cursor.execute(f"CREATE INDEX IF NOT EXISTS idx_score1 ON article_relevance_scores ('score_{weight_key_1}')")
            conn.commit()
            conn.close()
            print(f"  [+] Successfully saved {len(scores_df):,} contributing article scores to the database.")
        except Exception as e:
            print(f"  [!] WARNING: Could not save article scores to database: {e}")
    else:
        print("  [!] No articles contained the target seed terms. No scores generated.")

# Calculate final Contextual Relevance Scores (CRS) with Information Content (IC) Penalty
    print("\n<<< Calculating CRS... >>>")
    N_global = float(total_articles_in_range)

    final_node_weights_1 = {}
    for term, data in aggregator_1.items():
        if data['count'] > 0:
            mean_ars = data['sum'] / data['count']
            vol_multiplier = np.log10(data['count'] + 1)
            # data['count'] acts as both |P_i| and G_i for the temporal baseline
            ic_penalty = -np.log10(data['count'] / N_global)
            final_node_weights_1[term] = mean_ars * vol_multiplier * ic_penalty

    final_node_weights_2 = {}
    for term, data in aggregator_2.items():
        if data['count'] > 0:
            mean_ars = data['sum'] / data['count']
            vol_multiplier = np.log10(data['count'] + 1)
            ic_penalty = -np.log10(data['count'] / N_global)
            final_node_weights_2[term] = mean_ars * vol_multiplier * ic_penalty

    print("\n<<< Generating Final Output JSON File >>>")
    try:
        with open(input_nodes_file, 'r') as f:
            network_data = json.load(f)

        for node in network_data.get('elements', {}).get('nodes', []):
            term = node.get('data', {}).get(id_key)
            node['data'][final_key_1] = final_node_weights_1.get(term, 0.0)
            node['data'][final_key_2] = final_node_weights_2.get(term, 0.0)

        os.makedirs(os.path.dirname(output_nodes_file), exist_ok=True)
        with open(output_nodes_file, 'w') as f:
            json.dump(network_data, f, indent=2)

        print(f"  [+] Success! Final relevance JSON created at: {os.path.basename(output_nodes_file)}")

    except Exception as e:
        raise RuntimeError(f"ERROR generating final relevance JSON: {e}")